In [39]:
import pandas as pd  # импорты
import numpy as np  # код по стандартам pep8
import torch
import torch.nn as nn
import torch.optim as optim  # оптимизаторы
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from tqdm import tqdm  # прогресс бар
import os  # нужно для того если я прерву обучение чтобы оно не начиналось с нуля
from sklearn.feature_extraction.text import TfidfVectorizer  # TF-IDF идею взял с baseline.ipynb
# я так то многие идеи по этой задаче взял с baseline.ipynb и объединил с моим решением 1 задачи

In [ ]:
train_df = pd.read_parquet('train.parquet',
                           engine='fastparquet')  # загрузка данных с fastparquet потому что parquet с чем то конфликтовал
test_df = pd.read_parquet('test.parquet', engine='fastparquet')

for col in ['title1', 'description1', 'title2', 'description2',  # предобработка чтобы с этим было проще работать
            'parentname1', 'subjectname1', 'parentname2', 'subjectname2']:
    train_df[col] = train_df[col].fillna('')
    test_df[col] = test_df[col].fillna('')

train_df['text1'] = train_df['title1'] + ' ' + train_df['description1'] + ' ' + train_df['subjectname1'] + ' ' + \
                    train_df['parentname1']
train_df['text2'] = train_df['title2'] + ' ' + train_df['description2'] + ' ' + train_df['subjectname2'] + ' ' + \
                    train_df['parentname2']
test_df['text1'] = test_df['title1'] + ' ' + test_df['description1'] + ' ' + test_df['subjectname1'] + ' ' + test_df[
    'parentname1']
test_df['text2'] = test_df['title2'] + ' ' + test_df['description2'] + ' ' + test_df['subjectname2'] + ' ' + test_df[
    'parentname2']

train_df['same_subject'] = (train_df['subjectname1'] == train_df['subjectname2']).astype(int)  # дополнительные признаки
train_df['same_parent'] = (train_df['parentname1'] == train_df['parentname2']).astype(int)
test_df['same_subject'] = (test_df['subjectname1'] == test_df['subjectname2']).astype(int)
test_df['same_parent'] = (test_df['parentname1'] == test_df['parentname2']).astype(int)
# модель не умеет работать с текстом напрямую надо переводить его в числа
tokenizer = AutoTokenizer.from_pretrained(
    'cointegrated/rubert-tiny2')

In [ ]:
class PriceModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained('cointegrated/rubert-tiny2')  # легкая модель
        self.bert.gradient_checkpointing_enable()  # для экономии видеопамяти
        self.fc = nn.Sequential(
            nn.Linear(936, 384),
            nn.ReLU(),
            nn.Dropout(0.3),  # вероятность отключения нейронов
            nn.Linear(384, 192), # сжатие
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(192, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
            nn.Sigmoid()  # вероятность от 0 до 1
        )

    def forward(self, input_ids1, attention_mask1, input_ids2, attention_mask2):
        outputs1 = self.bert(input_ids=input_ids1,
                             attention_mask=attention_mask1)  # переход от того что думает модель к ответу
        last_hidden1 = outputs1.last_hidden_state[:, 0, :]
        outputs2 = self.bert(input_ids=input_ids2, attention_mask=attention_mask2)  # тоже самое
        last_hidden2 = outputs2.last_hidden_state[:, 0, :]
        bert_combined = torch.cat([last_hidden1, last_hidden2, torch.abs(last_hidden1 - last_hidden2)], dim=1)
        return self.fc(bert_combined).squeeze()

In [37]:
class PriceDataset(Dataset):
    def __init__(self, df, tokenizer, is_train=True):
        self.df = df  # инициализация
        self.tokenizer = tokenizer
        self.is_train = is_train

    def __len__(self):  # тут все понятно
        return len(self.df)

    def __getitem__(self, idx):  # получение элемента
        text1 = str(self.df.iloc[idx]['text1'])
        text2 = str(self.df.iloc[idx]['text2'])
        encoding1 = self.tokenizer(
            text1,
            truncation=True,
            padding='max_length',
            max_length=392,  # длина контекста
            return_tensors='pt'
        )
        encoding2 = self.tokenizer(
            text2,
            truncation=True,
            padding='max_length',
            max_length=392,  # тоже самое
            return_tensors='pt'
        )
        result = {
            'input_ids1': encoding1['input_ids'].squeeze(), # преобразования в числа модель не понимает текст
            'attention_mask1': encoding1['attention_mask'].squeeze(),
            'input_ids2': encoding2['input_ids'].squeeze(), 
            'attention_mask2': encoding2['attention_mask'].squeeze(),
        }
        if self.is_train:
            result['label'] = torch.tensor(self.df.iloc[idx]['is_duplicates'], dtype=torch.float32)  # метка дубликата
        return result

In [38]:
train_data, val_data = train_test_split(train_df, test_size=0.1,
                                        stratify=train_df['is_duplicates'],
                                        random_state=42)  # 90% на обучение остальное на валидацию
train_dataset = PriceDataset(train_data, tokenizer, is_train=True)  # для токенизирования текста
val_dataset = PriceDataset(val_data, tokenizer, is_train=True)  # для обучения и валидации
test_dataset = PriceDataset(test_df, tokenizer, is_train=False)  # для тестирования
train_loader = DataLoader(train_dataset, batch_size=6,
                          shuffle=True)  # батч такой из-за ограничений vram + перемешивание результатов
val_loader = DataLoader(val_dataset, batch_size=24, shuffle=False)  # тут не принципиален батч сайз(т.к. нет градиентов)
test_loader = DataLoader(test_dataset, batch_size=24, shuffle=False)  # нет перемешивания результатов

In [ ]:
device = torch.device('cuda')  # обучал на видеокарте
model = PriceModel().to(device)

if os.path.exists('best_model2.pth'):  # если есть модель то загружаю
    model.load_state_dict(torch.load('best_model2.pth'))

optimizer = optim.Adam(model.parameters(), lr=3e-6)  # ставил руками
criterion = nn.BCELoss()  # функция потерь
best_val_loss = 999999999  # ставил руками сначала такой

for epoch in range(100):  # не принципиально я часто останавливал обучение
    model.train()
    train_loss = 0  # обучение
    
    for batch in tqdm(train_loader, leave=False):  # прогресс бар для красоты
        input_ids1 = batch['input_ids1'].to(device)
        attention_mask1 = batch['attention_mask1'].to(device)
        input_ids2 = batch['input_ids2'].to(device)
        attention_mask2 = batch['attention_mask2'].to(device)
        labels = batch['label'].to(device) # праильные ответы
        optimizer.zero_grad() # обнуление градиентов 
        preds = model(input_ids1, attention_mask1, input_ids2, attention_mask2)
        loss = criterion(preds, labels) # подсчет ошибки
        loss.backward() # подсчет градиентов
        optimizer.step()# добавляет в веса
        train_loss += loss.item()  # подсчет ошибки
    model.eval()
    val_loss = 0
    with torch.no_grad():  # валидация
        for batch in val_loader: # без прогресс бара
            input_ids1 = batch['input_ids1'].to(device)
            attention_mask1 = batch['attention_mask1'].to(device)
            input_ids2 = batch['input_ids2'].to(device)
            attention_mask2 = batch['attention_mask2'].to(device)
            labels = batch['label'].to(device)
            preds = model(input_ids1, attention_mask1, input_ids2, attention_mask2)
            loss = criterion(preds, labels)
            val_loss += loss.item()  
    avg_train_loss = train_loss / len(train_loader)  # средние train и val loss
    avg_val_loss = val_loss / len(val_loader)
    
    print(f"train Loss {avg_train_loss:.4f}, val Loss {avg_val_loss:.4f}")  # вывод с точностью до 4 знаков
    if avg_val_loss < best_val_loss:  # сохраняю на случай если захочу дальше обучать
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_model2.pth')
        print("сохранена лучшая модель")
    else:
        break

In [ ]:
model.load_state_dict(torch.load('best_model2.pth')) # загрузка модели
model.eval()
predictions = []  # предсказания
with torch.no_grad(): # деоание предсказаний
    for batch in test_loader:  # решил без прогресс бара
        input_ids1 = batch['input_ids1'].to(device)
        attention_mask1 = batch['attention_mask1'].to(device)
        input_ids2 = batch['input_ids2'].to(device)
        attention_mask2 = batch['attention_mask2'].to(device)
        preds = model(input_ids1, attention_mask1, input_ids2, attention_mask2)
        predictions.extend(preds.cpu().numpy())
predictions = np.array(predictions)
results = pd.DataFrame({  # создание csv таблицы с ответами
    'id': test_df['id'].values,
    'y_pred': predictions
})
results.to_csv('submission.csv', index=False) # сохранение
print('все')